# Train — BYOL (GPU 1)

**실행 환경**: 이 노트북은 GPU 1이 할당된 Jupyter 인스턴스에서 실행한다.

터미널에서:
```bash
CUDA_VISIBLE_DEVICES=1 jupyter lab --port 8889
```

MoCo v2 노트북과 **동시에 실행** 가능 — 같은 ssl_lib을 import하지만 GPU/메모리는 완전 분리.

## Cell 1 — 환경 확인

In [ ]:
%load_ext autoreload
%autoreload 2

import torch

assert torch.cuda.is_available(), 'GPU 사용 불가!'
assert torch.cuda.device_count() == 1, (
    f'GPU {torch.cuda.device_count()}개 보임. '
    f'CUDA_VISIBLE_DEVICES=1로 노트북 띄웠는지 확인.'
)
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Config 로드

In [ ]:
import yaml

with open('../configs/byol_r50.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['root'] = '../data'
cfg['output']['dir'] = '../outputs/byol_r50_seed42'
cfg['output']['log_file'] = '../logs/byol_seed42.log'

print(yaml.dump(cfg, allow_unicode=True))

## Cell 3 — 학습 시작

**모니터링 포인트**: BYOL은 collapse 위험이 있으므로 `feature_std`를 매번 확인.
0.05 미만으로 떨어지면 collapse 시작 신호일 수 있음.

In [ ]:
from ssl_lib.train_loop import pretrain

pretrain(cfg)

## (선택) Resume

In [ ]:
# pretrain(cfg, resume_from='../outputs/byol_r50_seed42/ckpt_ep200.pth')